Step 1: Check Syetem and Load key

In [22]:
# @title Step 1: Cloud/Colab Runtime Setup & FIRMS API Key Loader
import os
import sys
import subprocess

REQUIRED_PACKAGES = ["folium", "geopandas", "shapely", "scipy", "matplotlib", "pyarrow", "requests"]
missing = []
for pkg in REQUIRED_PACKAGES:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"📦 Installing missing packages: {', '.join(missing)}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import pandas as pd
import geopandas as gpd
import scipy

INLINE_FIRMS_MAP_KEY = ""  # @param {type:"string"}

def load_firms_key(inline_key: str = "") -> str:
    # 1. Explicit inline input
    if inline_key and inline_key.strip():
        return inline_key.strip()
    # 2. Google Colab Secrets (userdata)
    try:
        from google.colab import userdata
        colab_key = userdata.get("FIRMS_MAP_KEY")
        if colab_key:
            return colab_key.strip()
    except Exception:
        pass
    # 3. Local .env or environment variable fallback
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    env_key = os.getenv("FIRMS_MAP_KEY", "").strip()
    if env_key:
        return env_key
    raise ValueError(
        "FIRMS_MAP_KEY not found! Add 'FIRMS_MAP_KEY' to Google Colab Secrets (🔑 icon on the left) "
        "or paste it into INLINE_FIRMS_MAP_KEY above."
    )

FIRMS_MAP_KEY = load_firms_key(INLINE_FIRMS_MAP_KEY)
print(f"✅ NASA FIRMS API key loaded ({len(FIRMS_MAP_KEY)} chars).")
print(f"✅ Runtime ready | Python {sys.version.split()[0]} | NumPy {np.__version__} | Pandas {pd.__version__} | GeoPandas {gpd.__version__} | SciPy {scipy.__version__}")

📦 Installing missing packages: scipy...
✅ NASA FIRMS API key loaded (32 chars).
✅ Runtime ready | Python 3.14.10 | NumPy 2.5.4 | Pandas 3.1.1 | GeoPandas 1.2.0 | SciPy 1.18.1


test GPU

In [ ]:
# @title

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)
    print("VRAM:", round(props.total_memory / (1024 ** 3), 2), "GB")
    print("CUDA version:", torch.version.cuda)

    # Quick GPU computation test
    x = torch.rand(3000, 3000, device="cuda")
    y = torch.rand(3000, 3000, device="cuda")
    z = x @ y

    print("GPU computation: SUCCESS")
    print("Tensor device:", z.device)
else:
    print("GPU not detected by PyTorch.")
    print("Check NVIDIA drivers and CUDA-enabled PyTorch installation.")


PyTorch version: 2.15.0+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 5.69 GB
CUDA version: 13.2
GPU computation: SUCCESS
Tensor device: cuda:0


In [ ]:
# @title Step 2: Drag and Drop Box on Map to Choose Region
import folium
from folium.plugins import Draw
from IPython.display import display

# Lock bounds to a single world copy so longitudes never exceed [-180, 180]
m = folium.Map(
    location=[23.5, 78.5],
    zoom_start=5,
    max_bounds=True,
    min_lon=-180,
    max_lon=180,
    min_lat=-90,
    max_lat=90,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
)

draw = Draw(
    export=False,
    position="topleft",
    draw_options={
        "polyline": False,
        "polygon": False,
        "circle": False,
        "marker": False,
        "circlemarker": False,
        "rectangle": {"shapeOptions": {"color": "#ff3333", "fillOpacity": 0.2}},
    },
    edit_options={"edit": True, "remove": True},
)
draw.add_to(m)

bbox_script = """
<div id="bbox-banner" style="
    position: fixed; bottom: 15px; left: 50%; transform: translateX(-50%);
    z-index: 9999; background: #111; color: #00f5d4; padding: 10px 18px;
    border-radius: 6px; font-family: monospace; font-size: 14px;
    border: 1px solid #00f5d4; box-shadow: 0 2px 10px rgba(0,0,0,0.5);">
    Click the ⬛ rectangle icon on the top-left, then click & drag on the map
</div>
<script>
document.addEventListener("DOMContentLoaded", function() {
    setTimeout(function() {
        var mapObj = Object.values(window).find(v => v instanceof L.Map);
        if (!mapObj) return;
        function wrapLon(lon) {
            return ((((lon + 180) % 360) + 360) % 360) - 180;
        }
        function updateBanner(layer) {
            var b = layer.getBounds();
            var w = wrapLon(b.getWest()).toFixed(4);
            var s = Math.max(-90, Math.min(90, b.getSouth())).toFixed(4);
            var e = wrapLon(b.getEast()).toFixed(4);
            var n = Math.max(-90, Math.min(90, b.getNorth())).toFixed(4);
            var bbox = w + "," + s + "," + e + "," + n;
            document.getElementById("bbox-banner").innerHTML =
                "Copied! Paste into Step 3 SELECTED_BBOX: <b>" + bbox + "</b>";
            if (navigator.clipboard) { navigator.clipboard.writeText(bbox); }
        }
        mapObj.on(L.Draw.Event.CREATED, function(e) { updateBanner(e.layer); });
        mapObj.on(L.Draw.Event.EDITED, function(e) {
            e.layers.eachLayer(function(layer) { updateBanner(layer); });
        });
    }, 500);
});
</script>
"""
m.get_root().html.add_child(folium.Element(bbox_script))
display(m)

In [ ]:
# @title Step 3: Fetch 10 Days (2x 5-Day Requests) & Inspect Sample Outputs
import io
import time
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone
from IPython.display import display

SELECTED_BBOX = "-122.1000,39.7000,-121.3000,40.4000"  # @param {type:"string"}
CHOSEN_DATE = "2024-07-26"  # @param {type:"date"}

SATELLITES = {
    "MODIS (Terra & Aqua)": {"NRT": "MODIS_NRT", "SP": "MODIS_SP"},
    "VIIRS NOAA-20":        {"NRT": "VIIRS_NOAA20_NRT", "SP": "VIIRS_NOAA20_SP"},
    "VIIRS NOAA-21":        {"NRT": "VIIRS_NOAA21_NRT", "SP": "VIIRS_NOAA21_SP"},
}

def sanitize_bbox(bbox_str: str) -> str:
    """Wraps longitudes into [-180, 180] and clamps latitudes to [-90, 90]."""
    w, s, e, n = map(float, bbox_str.strip().split(","))
    w = ((w + 180) % 360) - 180
    e = ((e + 180) % 360) - 180
    s = max(-90.0, min(90.0, s))
    n = max(-90.0, min(90.0, n))
    return f"{w:.4f},{s:.4f},{e:.4f},{n:.4f}"

def fetch_10_days_around_date(map_key: str, raw_bbox: str, chosen_date_str: str):
    bbox = sanitize_bbox(raw_bbox)
    center_dt = datetime.strptime(chosen_date_str, "%Y-%m-%d").date()
    today_utc = datetime.now(timezone.utc).date()

    # Request 1: 4 days before + chosen date = 5 days (D-4 to D)
    req1_start = center_dt - timedelta(days=4)
    req1_end   = center_dt

    # Request 2: 5 days after chosen date = 5 days (D+1 to D+5)
    req2_start = center_dt + timedelta(days=1)
    req2_end   = center_dt + timedelta(days=5)

    windows = [
        ("Request 1 (D-4 to D)", req1_start, req1_end, 5),
        ("Request 2 (D+1 to D+5)", req2_start, req2_end, 5),
    ]

    print(f"Target Region : {bbox}")
    print(f"Chosen Date   : {center_dt}")
    print(f"Full 10d Span : {req1_start} -> {req2_end}\n" + "=" * 75)

    sat_data = {}

    for sat_name, sources in SATELLITES.items():
        print(f"\n🛰️  {sat_name}")
        chunks = []

        for label, w_start, w_end, day_span in windows:
            mode = "SP" if (today_utc - w_end).days > 60 else "NRT"
            source_id = sources[mode]
            start_str = w_start.strftime("%Y-%m-%d")

            url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{source_id}/{bbox}/{day_span}/{start_str}"
            resp = requests.get(url, timeout=30)

            if resp.status_code == 200 and "latitude" in resp.text[:100]:
                df_part = pd.read_csv(io.StringIO(resp.text))
                chunks.append(df_part)
                print(f"  ✔ {label} [{w_start} to {w_end}] ({source_id}): {len(df_part)} rows")
            else:
                print(f"  ✖ {label} [{w_start} to {w_end}] ({source_id}): HTTP {resp.status_code} | {resp.text[:100].strip()}")

            time.sleep(0.25)

        combined = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
        if not combined.empty:
            combined["acq_time_utc"] = combined["acq_time"].astype(str).str.zfill(4)
            combined = combined.sort_values(["acq_date", "acq_time_utc"]).reset_index(drop=True)

        sat_data[sat_name] = combined
        print(f"  ➜ Total combined 10-day rows for {sat_name}: {len(combined)}")

        if not combined.empty:
            print(f"  📅 Date range returned: {combined['acq_date'].min()} to {combined['acq_date'].max()}")
            display(combined.head(3))
        else:
            print("  ⚠️ No fire detections found in this bounding box/date window.")

    return sat_data, bbox

SAT_DATA, CLEAN_BBOX = fetch_10_days_around_date(FIRMS_MAP_KEY, SELECTED_BBOX, CHOSEN_DATE)

Target Region : -122.1000,39.7000,-121.3000,40.4000
Chosen Date   : 2024-07-26
Full 10d Span : 2024-07-22 -> 2024-07-31

🛰️  MODIS (Terra & Aqua)
  ✔ Request 1 (D-4 to D) [2024-07-22 to 2024-07-26] (MODIS_SP): 1211 rows
  ✔ Request 2 (D+1 to D+5) [2024-07-27 to 2024-07-31] (MODIS_SP): 996 rows
  ➜ Total combined 10-day rows for MODIS (Terra & Aqua): 2207
  📅 Date range returned: 2024-07-24 to 2024-07-31


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type,acq_time_utc
0,39.7721,-121.7730,351.3,3.0,1.6,2024-07-24,2233,Aqua,MODIS,95,61.03,306.9,228.1,D,0,2233
1,39.7813,-121.7643,313.8,3.5,1.8,2024-07-25,438,Terra,MODIS,88,61.03,297.5,43.8,N,0,0438
2,39.7920,-121.7729,325.0,3.5,1.8,2024-07-25,438,Terra,MODIS,100,61.03,298.5,106.7,N,0,0438



🛰️  VIIRS NOAA-20
  ✔ Request 1 (D-4 to D) [2024-07-22 to 2024-07-26] (VIIRS_NOAA20_SP): 3390 rows
  ✔ Request 2 (D+1 to D+5) [2024-07-27 to 2024-07-31] (VIIRS_NOAA20_SP): 4077 rows
  ➜ Total combined 10-day rows for VIIRS NOAA-20: 7467
  📅 Date range returned: 2024-07-24 to 2024-07-31


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,acq_time_utc
0,39.76785,-121.76293,354.17,0.54,0.68,2024-07-24,2201,N20,VIIRS,n,2,298.46,5.77,D,0,2201
1,39.77399,-121.76971,333.14,0.54,0.68,2024-07-24,2201,N20,VIIRS,n,2,301.27,20.10,D,0,2201
2,39.77415,-121.76353,355.54,0.54,0.68,2024-07-24,2201,N20,VIIRS,n,2,303.51,21.85,D,0,2201



🛰️  VIIRS NOAA-21
  ✔ Request 1 (D-4 to D) [2024-07-22 to 2024-07-26] (VIIRS_NOAA21_SP): 3985 rows
  ✔ Request 2 (D+1 to D+5) [2024-07-27 to 2024-07-31] (VIIRS_NOAA21_SP): 4158 rows
  ➜ Total combined 10-day rows for VIIRS NOAA-21: 8143
  📅 Date range returned: 2024-07-25 to 2024-07-31


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,acq_time_utc
0,39.78107,-121.75391,352.10,0.45,0.47,2024-07-25,930,N21,VIIRS,n,2,299.61,9.94,N,0,0930
1,39.78152,-121.75896,352.19,0.45,0.47,2024-07-25,930,N21,VIIRS,n,2,300.27,6.74,N,0,0930
2,39.78198,-121.76425,313.28,0.45,0.47,2024-07-25,930,N21,VIIRS,n,2,296.68,6.74,N,0,0930


In [ ]:
# @title Step 4: Chronological Local-Date + Day/Night Overpass Comparison
import json
import html
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from IPython.display import display, HTML

USE_EXACT_SCAN_TRACK = True  # @param {type:"boolean"}

center_dt = datetime.strptime(CHOSEN_DATE, "%Y-%m-%d").date()
start_dt = center_dt - timedelta(days=4)
all_10_dates = [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(10)]
default_date_idx = all_10_dates.index(CHOSEN_DATE) if CHOSEN_DATE in all_10_dates else 4

w, s, e, n = map(float, CLEAN_BBOX.split(","))
center_lat, center_lon = (s + n) / 2.0, (w + e) / 2.0

NOMINAL_RES_M = {
    "MODIS (Terra & Aqua)": 1000.0,
    "VIIRS NOAA-20":        375.0,
    "VIIRS NOAA-21":        375.0,
}

all_frp_vals = [
    val
    for df in SAT_DATA.values()
    if not df.empty
    for val in df["frp"].dropna().tolist()
]
frp_min = min(all_frp_vals) if all_frp_vals else 0.0
frp_max = np.percentile(all_frp_vals, 95) if len(all_frp_vals) > 1 else (max(all_frp_vals) if all_frp_vals else 100.0)
if frp_max <= frp_min:
    frp_max = frp_min + 1.0

orange_cmap = plt.get_cmap("YlOrRd")

def frp_to_orange_hex(frp_val: float) -> str:
    norm = np.clip((frp_val - frp_min) / (frp_max - frp_min), 0.0, 1.0)
    return mcolors.to_hex(orange_cmap(0.25 + 0.70 * norm))

def confidence_to_opacity(conf_val) -> tuple[str, float]:
    s_val = str(conf_val).strip().lower()
    if s_val in ("l", "low"):
        return "Low", 0.30
    if s_val in ("n", "nominal"):
        return "Nominal", 0.62
    if s_val in ("h", "high"):
        return "High", 0.95
    try:
        num = float(s_val)
        if num < 30:
            return f"Low ({int(num)}%)", 0.30
        elif num < 80:
            return f"Nominal ({int(num)}%)", 0.62
        else:
            return f"High ({int(num)}%)", 0.95
    except ValueError:
        return "Unknown", 0.55

# Convert UTC timestamp to Local Fire Date (shifting by 4h after solar midnight so 1:30 AM stays with that night)
sat_payload = {}
for sat_name, df in SAT_DATA.items():
    res_m = NOMINAL_RES_M[sat_name]
    records = []
    if not df.empty:
        df_sorted = df.sort_values("frp", ascending=False).copy()
        utc_dt = pd.to_datetime(df_sorted["acq_date"] + " " + df_sorted["acq_time_utc"], format="%Y-%m-%d %H%M", utc=True)
        # Shift to local solar time, minus 4 hours so early-morning (~01:30) night passes belong to the evening they started
        solar_offset = pd.to_timedelta(df_sorted["longitude"] / 15.0 - 4.0, unit="h")
        df_sorted["local_fire_date"] = (utc_dt + solar_offset).dt.strftime("%Y-%m-%d")

        for _, row in df_sorted.iterrows():
            lat = float(row["latitude"])
            lon = float(row["longitude"])
            frp_val = float(row["frp"])
            conf_label, fill_op = confidence_to_opacity(row["confidence"])

            if USE_EXACT_SCAN_TRACK and "scan" in df_sorted.columns and "track" in df_sorted.columns:
                width_m = float(row["scan"]) * 1000.0
                height_m = float(row["track"]) * 1000.0
            else:
                width_m = res_m
                height_m = res_m

            half_dlat = (height_m / 111320.0) / 2.0
            half_dlon = (width_m / (111320.0 * max(np.cos(np.radians(lat)), 1e-6))) / 2.0

            records.append({
                "d": str(row["local_fire_date"]),
                "ud": str(row["acq_date"]),
                "t": str(row.get("acq_time_utc", "")),
                "dn": str(row.get("daynight", "D")).strip().upper(),
                "frp": round(frp_val, 2),
                "c": frp_to_orange_hex(frp_val),
                "o": fill_op,
                "cl": conf_label,
                "res": f"{int(width_m)}m x {int(height_m)}m",
                "b": [
                    [round(lat - half_dlat, 5), round(lon - half_dlon, 5)],
                    [round(lat + half_dlat, 5), round(lon + half_dlon, 5)],
                ],
            })
    sat_payload[sat_name] = records

html_app = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ background: #0e0e0e; color: #eee; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, monospace; padding: 10px; }}
    .legend-bar {{
      background: #161616; border: 1px solid #333; border-radius: 6px; padding: 8px 14px;
      font-size: 12px; margin-bottom: 10px; display: flex; flex-wrap: wrap;
      justify-content: space-between; align-items: center; gap: 10px;
    }}
    .main-stage {{
      display: grid;
      grid-template-columns: 1fr 84px;
      gap: 10px;
      align-items: stretch;
    }}
    .maps-grid {{
      display: grid;
      grid-template-columns: repeat(3, 1fr);
      gap: 10px;
    }}
    .map-card {{
      background: #161616; border: 1px solid #333; border-radius: 6px; padding: 8px;
      display: flex; flex-direction: column;
    }}
    .card-header {{
      display: flex; justify-content: space-between; align-items: center;
      font-size: 12px; font-weight: 600; margin-bottom: 6px;
    }}
    .map-box {{
      position: relative; width: 100%; aspect-ratio: 1 / 1;
      border-radius: 4px; overflow: hidden; background: #1e1e1e;
    }}
    .leaflet-map {{ width: 100%; height: 100%; }}
    .time-sidebar {{
      background: #161616; border: 1px solid #333; border-radius: 6px; padding: 10px 6px;
      display: flex; flex-direction: column; align-items: center; justify-content: space-between;
    }}
    .time-title {{ font-size: 11px; color: #00f5d4; font-weight: bold; text-align: center; }}
    .v-slider-wrap {{
      position: relative; height: 220px; width: 32px; display: flex; align-items: center; justify-content: center;
    }}
    input[type=range].v-slider {{
      -webkit-appearance: none; appearance: none;
      width: 200px; height: 8px; background: #2c2c2c; border-radius: 4px; outline: none;
      transform: rotate(-90deg); cursor: pointer;
    }}
    input[type=range].v-slider::-webkit-slider-thumb {{
      -webkit-appearance: none; appearance: none;
      width: 18px; height: 18px; border-radius: 50%; background: #00f5d4; cursor: pointer; border: 2px solid #111;
    }}
    .time-labels {{
      display: flex; flex-direction: column; justify-content: space-between;
      height: 190px; font-size: 10px; text-align: left; color: #aaa; pointer-events: none;
    }}
    .time-badge {{
      background: #222; border: 1px solid #00f5d4; color: #00f5d4;
      font-size: 10px; padding: 3px 5px; border-radius: 4px; text-align: center; width: 100%;
    }}
    .date-footer {{
      margin-top: 10px; background: #161616; border: 1px solid #333; border-radius: 6px;
      padding: 10px 16px;
    }}
    .date-top {{
      display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px; font-size: 13px;
    }}
    input[type=range].h-slider {{
      -webkit-appearance: none; appearance: none;
      width: 100%; height: 8px; background: #2c2c2c; border-radius: 4px; outline: none; cursor: pointer;
    }}
    input[type=range].h-slider::-webkit-slider-thumb {{
      -webkit-appearance: none; appearance: none;
      width: 18px; height: 18px; border-radius: 50%; background: #ff9f1c; cursor: pointer; border: 2px solid #111;
    }}
    .date-ticks {{
      display: flex; justify-content: space-between; font-size: 10px; color: #888; margin-top: 5px;
    }}
    .date-ticks span.chosen {{ color: #ff9f1c; font-weight: bold; }}
  </style>
</head>
<body>

  <div class="legend-bar">
    <div>
      <b id="active-banner" style="color:#ff9f1c;">Local Fire Day: {CHOSEN_DATE} | Pass: All (24h)</b> &nbsp;|&nbsp;
      <span>FRP: </span>
      <span style="background:linear-gradient(90deg,#fed976,#fd8d3c,#e31a1c);padding:1px 24px;border-radius:3px;"></span>
      <span> {frp_min:.1f} → {frp_max:.1f}+ MW</span>
    </div>
    <div>
      <span>Confidence Opacity: </span>
      <span style="display:inline-block;width:11px;height:11px;background:#fd8d3c;opacity:0.30;border:1px solid #fff;vertical-align:middle;"></span> Low &nbsp;
      <span style="display:inline-block;width:11px;height:11px;background:#fd8d3c;opacity:0.62;border:1px solid #fff;vertical-align:middle;"></span> Nominal &nbsp;
      <span style="display:inline-block;width:11px;height:11px;background:#fd8d3c;opacity:0.95;border:1px solid #fff;vertical-align:middle;"></span> High
    </div>
  </div>

  <div class="main-stage">
    <div class="maps-grid">
      <div class="map-card">
        <div class="card-header">
          <span>MODIS (Terra &amp; Aqua)</span>
          <span id="stat-0" style="color:#ff9f1c;">0 pts | 0 MW</span>
        </div>
        <div class="map-box"><div id="map-0" class="leaflet-map"></div></div>
      </div>

      <div class="map-card">
        <div class="card-header">
          <span>VIIRS NOAA-20</span>
          <span id="stat-1" style="color:#ff9f1c;">0 pts | 0 MW</span>
        </div>
        <div class="map-box"><div id="map-1" class="leaflet-map"></div></div>
      </div>

      <div class="map-card">
        <div class="card-header">
          <span>VIIRS NOAA-21</span>
          <span id="stat-2" style="color:#ff9f1c;">0 pts | 0 MW</span>
        </div>
        <div class="map-box"><div id="map-2" class="leaflet-map"></div></div>
      </div>
    </div>

    <div class="time-sidebar">
      <div class="time-title">OVERPASS<br>CYCLE</div>
      <div style="display:flex;align-items:center;gap:4px;">
        <div class="v-slider-wrap">
          <input id="time-slider" class="v-slider" type="range" min="0" max="2" step="1" value="1" />
        </div>
        <div class="time-labels">
          <span>☀ Day<br><small>(1st)</small></span>
          <span>🌐 All<br><small>(24h)</small></span>
          <span>🌙 Night<br><small>(2nd)</small></span>
        </div>
      </div>
      <div id="time-badge" class="time-badge">All (24h)</div>
    </div>
  </div>

  <div class="date-footer">
    <div class="date-top">
      <span>📅 <b>Local Solar Fire Day</b> (Groups each day's afternoon pass with that same night)</span>
      <span id="date-readout" style="color:#ff9f1c;font-weight:bold;">{CHOSEN_DATE}</span>
    </div>
    <input id="date-slider" class="h-slider" type="range" min="0" max="9" step="1" value="{default_date_idx}" />
    <div class="date-ticks">
      {"".join([f'<span class="{"chosen" if d == CHOSEN_DATE else ""}">{d[5:]}</span>' for d in all_10_dates])}
    </div>
  </div>

<script>
  const DATES = {json.dumps(all_10_dates)};
  const SAT_KEYS = ["MODIS (Terra & Aqua)", "VIIRS NOAA-20", "VIIRS NOAA-21"];
  const SAT_DATA = {json.dumps(sat_payload)};
  const BBOX = [[{s}, {w}], [{n}, {e}]];

  const maps = [];
  const layerGroups = [];
  let isSyncing = false;

  SAT_KEYS.forEach((key, idx) => {{
    const m = L.map("map-" + idx, {{
      center: [{center_lat}, {center_lon}],
      zoom: 8,
      preferCanvas: true
    }});
    L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{{z}}/{{y}}/{{x}}", {{
      attribution: "Esri Dark Gray"
    }}).addTo(m);

    L.rectangle(BBOX, {{ color: "#888888", weight: 1.5, fill: false }}).addTo(m);
    m.fitBounds(BBOX);

    const lg = L.layerGroup().addTo(m);
    maps.push(m);
    layerGroups.push(lg);

    m.on("moveend", () => {{
      if (isSyncing) return;
      isSyncing = true;
      const c = m.getCenter();
      const z = m.getZoom();
      maps.forEach((other, j) => {{
        if (idx !== j) other.setView(c, z, {{ animate: false }});
      }});
      isSyncing = false;
    }});
  }});

  const dateSlider = document.getElementById("date-slider");
  const timeSlider = document.getElementById("time-slider");

  function renderAll() {{
    const selectedDate = DATES[parseInt(dateSlider.value, 10)];
    const timeModeVal = parseInt(timeSlider.value, 10);
    const timeLabel = timeModeVal === 2 ? "Daytime Pass (D)" : (timeModeVal === 0 ? "Same-Night Pass (N)" : "All Passes (24h)");

    document.getElementById("date-readout").textContent = selectedDate;
    document.getElementById("time-badge").textContent = timeModeVal === 2 ? "Day (D)" : (timeModeVal === 0 ? "Night (N)" : "All (24h)");
    document.getElementById("active-banner").textContent = `Local Fire Day: ${{selectedDate}} | Pass: ${{timeLabel}}`;

    SAT_KEYS.forEach((key, idx) => {{
      const lg = layerGroups[idx];
      lg.clearLayers();

      const rows = SAT_DATA[key] || [];
      let count = 0;
      let totalFrp = 0;

      for (let i = 0; i < rows.length; i++) {{
        const r = rows[i];
        if (r.d !== selectedDate) continue;
        if (timeModeVal === 2 && r.dn !== "D") continue;
        if (timeModeVal === 0 && r.dn !== "N") continue;

        count++;
        totalFrp += r.frp;

        if (count <= 3000) {{
          L.rectangle(r.b, {{
            color: r.c,
            weight: 0.5,
            opacity: r.o,
            fillColor: r.c,
            fillOpacity: r.o
          }}).bindPopup(
            `<b>${{key}}</b><br>Local Fire Day: ${{r.d}}<br>Raw UTC: ${{r.ud}} ${{r.t}} (${{r.dn}})<br>FRP: <b>${{r.frp}} MW</b><br>Conf: <b>${{r.cl}}</b><br>Pixel: ${{r.res}}`
          ).addTo(lg);
        }}
      }}

      document.getElementById("stat-" + idx).textContent =
        `${{count.toLocaleString()}} pts | ${{Math.round(totalFrp).toLocaleString()}} MW`;
    }});
  }}

  dateSlider.addEventListener("input", renderAll);
  timeSlider.addEventListener("input", renderAll);
  renderAll();
</script>
</body>
</html>
"""

display(HTML(f'<iframe srcdoc="{html.escape(html_app)}" style="width:100%;height:610px;border:none;border-radius:8px;"></iframe>'))

/home/tinted/Projects/NASA-Space-Apps-2026/.venv/lib/python3.14/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [ ]:
# @title Step 4 -  take 2 - different view : 3 Square Maps + Unified Chronological Day/Night Timeline
import json
import html
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from IPython.display import display, HTML

USE_EXACT_SCAN_TRACK = True  # @param {type:"boolean"}

center_dt = datetime.strptime(CHOSEN_DATE, "%Y-%m-%d").date()
start_dt = center_dt - timedelta(days=4)
all_10_dates = [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d") for i in range(10)]

# Build 20 chronological timeline stops: [Date 1 Day, Date 1 Night, Date 2 Day, Date 2 Night, ...]
timeline_slots = []
for d in all_10_dates:
    timeline_slots.append({"date": d, "dn": "D", "label": f"{d} | ☀ Day",   "short": f"{d[5:]}\n☀"})
    timeline_slots.append({"date": d, "dn": "N", "label": f"{d} | 🌙 Night", "short": "🌙"})

default_slot_idx = all_10_dates.index(CHOSEN_DATE) * 2 if CHOSEN_DATE in all_10_dates else 8

w, s, e, n = map(float, CLEAN_BBOX.split(","))
center_lat, center_lon = (s + n) / 2.0, (w + e) / 2.0

NOMINAL_RES_M = {
    "MODIS (Terra & Aqua)": 1000.0,
    "VIIRS NOAA-20":        375.0,
    "VIIRS NOAA-21":        375.0,
}

all_frp_vals = [
    val
    for df in SAT_DATA.values()
    if not df.empty
    for val in df["frp"].dropna().tolist()
]
frp_min = min(all_frp_vals) if all_frp_vals else 0.0
frp_max = np.percentile(all_frp_vals, 95) if len(all_frp_vals) > 1 else (max(all_frp_vals) if all_frp_vals else 100.0)
if frp_max <= frp_min:
    frp_max = frp_min + 1.0

orange_cmap = plt.get_cmap("YlOrRd")

def frp_to_orange_hex(frp_val: float) -> str:
    norm = np.clip((frp_val - frp_min) / (frp_max - frp_min), 0.0, 1.0)
    return mcolors.to_hex(orange_cmap(0.25 + 0.70 * norm))

def confidence_to_opacity(conf_val) -> tuple[str, float]:
    s_val = str(conf_val).strip().lower()
    if s_val in ("l", "low"):
        return "Low", 0.30
    if s_val in ("n", "nominal"):
        return "Nominal", 0.62
    if s_val in ("h", "high"):
        return "High", 0.95
    try:
        num = float(s_val)
        if num < 30:
            return f"Low ({int(num)}%)", 0.30
        elif num < 80:
            return f"Nominal ({int(num)}%)", 0.62
        else:
            return f"High ({int(num)}%)", 0.95
    except ValueError:
        return "Unknown", 0.55

# Align UTC timestamps to Local Solar Fire Day so Day (D) -> Night (N) flows chronologically
sat_payload = {}
for sat_name, df in SAT_DATA.items():
    res_m = NOMINAL_RES_M[sat_name]
    records = []
    if not df.empty:
        df_sorted = df.sort_values("frp", ascending=False).copy()
        utc_dt = pd.to_datetime(df_sorted["acq_date"] + " " + df_sorted["acq_time_utc"], format="%Y-%m-%d %H%M", utc=True)
        solar_offset = pd.to_timedelta(df_sorted["longitude"] / 15.0 - 4.0, unit="h")
        df_sorted["local_fire_date"] = (utc_dt + solar_offset).dt.strftime("%Y-%m-%d")

        for _, row in df_sorted.iterrows():
            lat = float(row["latitude"])
            lon = float(row["longitude"])
            frp_val = float(row["frp"])
            conf_label, fill_op = confidence_to_opacity(row["confidence"])

            if USE_EXACT_SCAN_TRACK and "scan" in df_sorted.columns and "track" in df_sorted.columns:
                width_m = float(row["scan"]) * 1000.0
                height_m = float(row["track"]) * 1000.0
            else:
                width_m = res_m
                height_m = res_m

            half_dlat = (height_m / 111320.0) / 2.0
            half_dlon = (width_m / (111320.0 * max(np.cos(np.radians(lat)), 1e-6))) / 2.0

            records.append({
                "d": str(row["local_fire_date"]),
                "ud": str(row["acq_date"]),
                "t": str(row.get("acq_time_utc", "")),
                "dn": str(row.get("daynight", "D")).strip().upper(),
                "frp": round(frp_val, 2),
                "c": frp_to_orange_hex(frp_val),
                "o": fill_op,
                "cl": conf_label,
                "res": f"{int(width_m)}m x {int(height_m)}m",
                "b": [
                    [round(lat - half_dlat, 5), round(lon - half_dlon, 5)],
                    [round(lat + half_dlat, 5), round(lon + half_dlon, 5)],
                ],
            })
    sat_payload[sat_name] = records

tick_spans = []
for idx, slot in enumerate(timeline_slots):
    if slot["dn"] == "D":
        cls = "tick-day chosen" if slot["date"] == CHOSEN_DATE else "tick-day"
        tick_spans.append(f'<span class="{cls}" onclick="setSlot({idx})">{slot["date"][5:]}<br>☀</span>')
    else:
        tick_spans.append(f'<span class="tick-night" onclick="setSlot({idx})">·<br>🌙</span>')

html_app = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ background: #0e0e0e; color: #eee; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, monospace; padding: 10px; }}
    .legend-bar {{
      background: #161616; border: 1px solid #333; border-radius: 6px; padding: 8px 14px;
      font-size: 12px; margin-bottom: 10px; display: flex; flex-wrap: wrap;
      justify-content: space-between; align-items: center; gap: 10px;
    }}
    .maps-grid {{
      display: grid;
      grid-template-columns: repeat(3, 1fr);
      gap: 10px;
      width: 100%;
    }}
    .map-card {{
      background: #161616; border: 1px solid #333; border-radius: 6px; padding: 8px;
      display: flex; flex-direction: column;
    }}
    .card-header {{
      display: flex; justify-content: space-between; align-items: center;
      font-size: 12px; font-weight: 600; margin-bottom: 6px;
    }}
    .map-box {{
      position: relative; width: 100%; aspect-ratio: 1 / 1;
      border-radius: 4px; overflow: hidden; background: #1e1e1e;
    }}
    .leaflet-map {{ width: 100%; height: 100%; }}

    /* Unified Bottom Chronological Day/Night Timeline */
    .timeline-footer {{
      margin-top: 10px; background: #161616; border: 1px solid #333; border-radius: 6px;
      padding: 10px 16px;
    }}
    .timeline-top {{
      display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px; font-size: 13px;
    }}
    input[type=range].h-slider {{
      -webkit-appearance: none; appearance: none;
      width: 100%; height: 8px; background: #2c2c2c; border-radius: 4px; outline: none; cursor: pointer;
    }}
    input[type=range].h-slider::-webkit-slider-thumb {{
      -webkit-appearance: none; appearance: none;
      width: 18px; height: 18px; border-radius: 50%; background: #ff9f1c; cursor: pointer; border: 2px solid #111;
    }}
    .timeline-ticks {{
      display: grid;
      grid-template-columns: repeat(20, 1fr);
      text-align: center;
      font-size: 10px;
      margin-top: 6px;
      user-select: none;
    }}
    .timeline-ticks span {{ cursor: pointer; padding: 2px 0; border-radius: 3px; line-height: 1.25; }}
    .timeline-ticks span:hover {{ background: #262626; }}
    .tick-day {{ color: #ddd; }}
    .tick-day.chosen {{ color: #ff9f1c; font-weight: bold; }}
    .tick-night {{ color: #7aa2f7; }}
  </style>
</head>
<body>

  <div class="legend-bar">
    <div>
      <b id="active-banner" style="color:#ff9f1c;">{timeline_slots[default_slot_idx]["label"]}</b> &nbsp;|&nbsp;
      <span>FRP: </span>
      <span style="background:linear-gradient(90deg,#fed976,#fd8d3c,#e31a1c);padding:1px 24px;border-radius:3px;"></span>
      <span> {frp_min:.1f} → {frp_max:.1f}+ MW</span>
    </div>
    <div>
      <span>Confidence Opacity: </span>
      <span style="display:inline-block;width:11px;height:11px;background:#fd8d3c;opacity:0.30;border:1px solid #fff;vertical-align:middle;"></span> Low &nbsp;
      <span style="display:inline-block;width:11px;height:11px;background:#fd8d3c;opacity:0.62;border:1px solid #fff;vertical-align:middle;"></span> Nominal &nbsp;
      <span style="display:inline-block;width:11px;height:11px;background:#fd8d3c;opacity:0.95;border:1px solid #fff;vertical-align:middle;"></span> High
    </div>
  </div>

  <div class="maps-grid">
    <div class="map-card">
      <div class="card-header">
        <span>MODIS (Terra &amp; Aqua)</span>
        <span id="stat-0" style="color:#ff9f1c;">0 pts | 0 MW</span>
      </div>
      <div class="map-box"><div id="map-0" class="leaflet-map"></div></div>
    </div>

    <div class="map-card">
      <div class="card-header">
        <span>VIIRS NOAA-20</span>
        <span id="stat-1" style="color:#ff9f1c;">0 pts | 0 MW</span>
      </div>
      <div class="map-box"><div id="map-1" class="leaflet-map"></div></div>
    </div>

    <div class="map-card">
      <div class="card-header">
        <span>VIIRS NOAA-21</span>
        <span id="stat-2" style="color:#ff9f1c;">0 pts | 0 MW</span>
      </div>
      <div class="map-box"><div id="map-2" class="leaflet-map"></div></div>
    </div>
  </div>

  <div class="timeline-footer">
    <div class="timeline-top">
      <span>🛰️ <b>Chronological Overpass Timeline</b> (Step through Day ☀ &amp; Night 🌙 passes across dates)</span>
      <span id="slot-readout" style="color:#ff9f1c;font-weight:bold;">{timeline_slots[default_slot_idx]["label"]}</span>
    </div>
    <input id="timeline-slider" class="h-slider" type="range" min="0" max="19" step="1" value="{default_slot_idx}" />
    <div class="timeline-ticks">
      {"".join(tick_spans)}
    </div>
  </div>

<script>
  const SLOTS = {json.dumps(timeline_slots)};
  const SAT_KEYS = ["MODIS (Terra & Aqua)", "VIIRS NOAA-20", "VIIRS NOAA-21"];
  const SAT_DATA = {json.dumps(sat_payload)};
  const BBOX = [[{s}, {w}], [{n}, {e}]];

  const maps = [];
  const layerGroups = [];
  let isSyncing = false;

  SAT_KEYS.forEach((key, idx) => {{
    const m = L.map("map-" + idx, {{
      center: [{center_lat}, {center_lon}],
      zoom: 8,
      preferCanvas: true
    }});
    L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{{z}}/{{y}}/{{x}}", {{
      attribution: "Esri Dark Gray"
    }}).addTo(m);

    L.rectangle(BBOX, {{ color: "#888888", weight: 1.5, fill: false }}).addTo(m);
    m.fitBounds(BBOX);

    const lg = L.layerGroup().addTo(m);
    maps.push(m);
    layerGroups.push(lg);

    m.on("moveend", () => {{
      if (isSyncing) return;
      isSyncing = true;
      const c = m.getCenter();
      const z = m.getZoom();
      maps.forEach((other, j) => {{
        if (idx !== j) other.setView(c, z, {{ animate: false }});
      }});
      isSyncing = false;
    }});
  }});

  const slider = document.getElementById("timeline-slider");

  function setSlot(idx) {{
    slider.value = idx;
    renderAll();
  }}

  function renderAll() {{
    const slot = SLOTS[parseInt(slider.value, 10)];
    document.getElementById("slot-readout").textContent = slot.label;
    document.getElementById("active-banner").textContent = slot.label;

    SAT_KEYS.forEach((key, idx) => {{
      const lg = layerGroups[idx];
      lg.clearLayers();

      const rows = SAT_DATA[key] || [];
      let count = 0;
      let totalFrp = 0;

      for (let i = 0; i < rows.length; i++) {{
        const r = rows[i];
        if (r.d !== slot.date || r.dn !== slot.dn) continue;

        count++;
        totalFrp += r.frp;

        if (count <= 3000) {{
          L.rectangle(r.b, {{
            color: r.c,
            weight: 0.5,
            opacity: r.o,
            fillColor: r.c,
            fillOpacity: r.o
          }}).bindPopup(
            `<b>${{key}}</b><br>Local Fire Day: ${{r.d}} (${{r.dn === "D" ? "Day" : "Night"}})<br>Raw UTC: ${{r.ud}} ${{r.t}}<br>FRP: <b>${{r.frp}} MW</b><br>Conf: <b>${{r.cl}}</b><br>Pixel: ${{r.res}}`
          ).addTo(lg);
        }}
      }}

      document.getElementById("stat-" + idx).textContent =
        `${{count.toLocaleString()}} pts | ${{Math.round(totalFrp).toLocaleString()}} MW`;
    }});
  }}

  slider.addEventListener("input", renderAll);
  renderAll();
</script>
</body>
</html>
"""

display(HTML(f'<iframe srcdoc="{html.escape(html_app)}" style="width:100%;height:610px;border:none;border-radius:8px;"></iframe>'))

/home/tinted/Projects/NASA-Space-Apps-2026/.venv/lib/python3.14/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


So how large of a area could i select and how large of a date range can i select ?

In [18]:
# @title Step 5: Stress-Test Area Size & Date Range Limits (Week / Month / Multi-Month)
import io
import time
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from IPython.display import display

TEST_START_DATE = "2024-07-01"  # @param {type:"date"}
RANGE_PRESET = "1 Month (30 days)"  # @param ["1 Week (7 days)", "1 Month (30 days)", "2 Months (60 days)"]

total_days = 7 if "7" in RANGE_PRESET else (30 if "30" in RANGE_PRESET else 60)
w, s, e, n = map(float, CLEAN_BBOX.split(","))

# Calculate approximate physical area in km²
lat_mid = np.radians((s + n) / 2.0)
height_km = abs(n - s) * 111.32
width_km = abs(e - w) * 111.32 * np.cos(lat_mid)
area_km2 = width_km * height_km

# Split into <= 5-day chunks per NASA FIRMS API rule
start_dt = datetime.strptime(TEST_START_DATE, "%Y-%m-%d").date()
end_dt = start_dt + timedelta(days=total_days - 1)
today_utc = datetime.now(timezone.utc).date()

chunks = []
curr = start_dt
while curr <= end_dt:
    c_end = min(curr + timedelta(days=4), end_dt)
    span = (c_end - curr).days + 1
    api_mode = "SP" if (today_utc - c_end).days > 60 else "NRT"
    chunks.append((curr.strftime("%Y-%m-%d"), span, api_mode))
    curr = c_end + timedelta(days=1)

tx_cost = total_days * 3  # 1 day x 1 sensor = 1 transaction

print(f"📐 Selected Region : {CLEAN_BBOX} (~{width_km:,.0f} km × {height_km:,.0f} km = {area_km2:,.0f} km²)")
print(f"📅 Date Window     : {start_dt} to {end_dt} ({total_days} days | {len(chunks)} chunks/sensor)")
print(f"🎟️ NASA Quota Cost : {tx_cost} transactions ({(tx_cost / 5000) * 100:.1f}% of your 5,000 / 10-min limit)\n" + "=" * 75)

SATELLITES = {
    "MODIS (Terra & Aqua)": {"NRT": "MODIS_NRT", "SP": "MODIS_SP"},
    "VIIRS NOAA-20":        {"NRT": "VIIRS_NOAA20_NRT", "SP": "VIIRS_NOAA20_SP"},
    "VIIRS NOAA-21":        {"NRT": "VIIRS_NOAA21_NRT", "SP": "VIIRS_NOAA21_SP"},
}

BULK_DATA = {}
perf_rows = []

for sat_name, sources in SATELLITES.items():
    t0 = time.perf_counter()
    parts = []
    for d_str, span, api_mode in chunks:
        url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_MAP_KEY}/{sources[api_mode]}/{CLEAN_BBOX}/{span}/{d_str}"
        r = requests.get(url, timeout=30)
        if r.status_code == 200 and "latitude" in r.text[:100]:
            df_c = pd.read_csv(io.StringIO(r.text))
            if not df_c.empty:
                parts.append(df_c)
        time.sleep(0.15)

    elapsed = time.perf_counter() - t0
    df_all = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    BULK_DATA[sat_name] = df_all

    # Browser rendering verdict for your web app
    if len(df_all) <= 3000:
        render_status = "✅ Smooth (Raw Pixels)"
    elif len(df_all) <= 15000:
        render_status = "⚠️ Heavy (Use Canvas / 1km Grid)"
    else:
        render_status = "🛑 Too Dense (Must Aggregate / Cap Area)"

    perf_rows.append({
        "Satellite": sat_name,
        "Total Points": len(df_all),
        "Active Days": df_all["acq_date"].nunique() if not df_all.empty else 0,
        "Fetch Time (s)": round(elapsed, 2),
        "Total FRP (MW)": round(df_all["frp"].sum(), 1) if not df_all.empty else 0.0,
        "App Render Verdict": render_status,
    })

display(pd.DataFrame(perf_rows))

📐 Selected Region : -122.1000,39.7000,-121.3000,40.4000 (~68 km × 78 km = 5,312 km²)
📅 Date Window     : 2024-07-01 to 2024-07-30 (30 days | 6 chunks/sensor)
🎟️ NASA Quota Cost : 90 transactions (1.8% of your 5,000 / 10-min limit)


,Satellite,Total Points,Active Days,Fetch Time (s),Total FRP (MW),App Render Verdict
0,MODIS (Terra & Aqua),2134,10,11.79,630571.1,✅ Smooth (Raw Pixels)
1,VIIRS NOAA-20,7098,10,15.96,234389.8,⚠️ Heavy (Use Canvas / 1km Grid)
2,VIIRS NOAA-21,7792,9,14.33,205259.5,⚠️ Heavy (Use Canvas / 1km Grid)


In [19]:
# @title Step 6: Combined Month / Period View (All Dates & Points Merged Side-by-Side)
import json
import html
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from IPython.display import display, HTML

USE_EXACT_SCAN_TRACK = True  # @param {type:"boolean"}

w, s, e, n = map(float, CLEAN_BBOX.split(","))
center_lat, center_lon = (s + n) / 2.0, (w + e) / 2.0

NOMINAL_RES_M = {
    "MODIS (Terra & Aqua)": 1000.0,
    "VIIRS NOAA-20":        375.0,
    "VIIRS NOAA-21":        375.0,
}

# Find all YYYY-MM months present in BULK_DATA
available_months = sorted({
    str(d)[:7]
    for df in BULK_DATA.values()
    if not df.empty
    for d in df["acq_date"].unique()
})
if not available_months:
    available_months = ["ALL"]

all_frp_vals = [
    val
    for df in BULK_DATA.values()
    if not df.empty
    for val in df["frp"].dropna().tolist()
]
frp_min = min(all_frp_vals) if all_frp_vals else 0.0
frp_max = np.percentile(all_frp_vals, 95) if len(all_frp_vals) > 1 else (max(all_frp_vals) if all_frp_vals else 100.0)
if frp_max <= frp_min:
    frp_max = frp_min + 1.0

orange_cmap = plt.get_cmap("YlOrRd")

def frp_to_orange_hex(frp_val: float) -> str:
    norm = np.clip((frp_val - frp_min) / (frp_max - frp_min), 0.0, 1.0)
    return mcolors.to_hex(orange_cmap(0.25 + 0.70 * norm))

def confidence_to_opacity(conf_val) -> tuple[str, float]:
    s_val = str(conf_val).strip().lower()
    if s_val in ("l", "low"):
        return "Low", 0.30
    if s_val in ("n", "nominal"):
        return "Nominal", 0.62
    if s_val in ("h", "high"):
        return "High", 0.95
    try:
        num = float(s_val)
        if num < 30:
            return f"Low ({int(num)}%)", 0.30
        elif num < 80:
            return f"Nominal ({int(num)}%)", 0.62
        else:
            return f"High ({int(num)}%)", 0.95
    except ValueError:
        return "Unknown", 0.55

sat_payload = {}
for sat_name, df in BULK_DATA.items():
    res_m = NOMINAL_RES_M[sat_name]
    records = []
    if not df.empty:
        # Sort ascending by FRP so highest-intensity hotspots draw on top
        df_sorted = df.sort_values("frp", ascending=True)
        for _, row in df_sorted.iterrows():
            lat = float(row["latitude"])
            lon = float(row["longitude"])
            frp_val = float(row["frp"])
            conf_label, fill_op = confidence_to_opacity(row["confidence"])

            if USE_EXACT_SCAN_TRACK and "scan" in df_sorted.columns and "track" in df_sorted.columns:
                width_m = float(row["scan"]) * 1000.0
                height_m = float(row["track"]) * 1000.0
            else:
                width_m = res_m
                height_m = res_m

            half_dlat = (height_m / 111320.0) / 2.0
            half_dlon = (width_m / (111320.0 * max(np.cos(np.radians(lat)), 1e-6))) / 2.0

            records.append({
                "m": str(row["acq_date"])[:7],
                "d": str(row["acq_date"]),
                "frp": round(frp_val, 2),
                "c": frp_to_orange_hex(frp_val),
                "o": fill_op,
                "cl": conf_label,
                "b": [
                    [round(lat - half_dlat, 5), round(lon - half_dlon, 5)],
                    [round(lat + half_dlat, 5), round(lon + half_dlon, 5)],
                ],
            })
    sat_payload[sat_name] = records

month_options_html = '<option value="ALL">All Fetched Dates Combined</option>' + "".join(
    [f'<option value="{m}">Month: {m} (All Days Merged)</option>' for m in available_months if m != "ALL"]
)

html_app = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ background: #0e0e0e; color: #eee; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, monospace; padding: 10px; }}
    .top-bar {{
      background: #161616; border: 1px solid #333; border-radius: 6px; padding: 8px 14px;
      font-size: 12px; margin-bottom: 10px; display: flex; flex-wrap: wrap;
      justify-content: space-between; align-items: center; gap: 10px;
    }}
    select {{
      background: #222; color: #ff9f1c; border: 1px solid #ff9f1c; padding: 4px 10px;
      border-radius: 4px; font-family: monospace; font-size: 12px; font-weight: bold; cursor: pointer;
    }}
    .maps-grid {{
      display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; width: 100%;
    }}
    .map-card {{
      background: #161616; border: 1px solid #333; border-radius: 6px; padding: 8px;
      display: flex; flex-direction: column;
    }}
    .card-header {{
      display: flex; justify-content: space-between; align-items: center;
      font-size: 12px; font-weight: 600; margin-bottom: 6px;
    }}
    .map-box {{
      position: relative; width: 100%; aspect-ratio: 1 / 1;
      border-radius: 4px; overflow: hidden; background: #1e1e1e;
    }}
    .leaflet-map {{ width: 100%; height: 100%; }}
  </style>
</head>
<body>

  <div class="top-bar">
    <div>
      <span>🗓️ View Filter: </span>
      <select id="month-select">{month_options_html}</select>
    </div>
    <div>
      <span>FRP: </span>
      <span style="background:linear-gradient(90deg,#fed976,#fd8d3c,#e31a1c);padding:1px 24px;border-radius:3px;"></span>
      <span> {frp_min:.1f} → {frp_max:.1f}+ MW</span>
      &nbsp;|&nbsp;
      <span>Opacity = Confidence (Low 30% / Nom 62% / High 95%)</span>
    </div>
  </div>

  <div class="maps-grid">
    <div class="map-card">
      <div class="card-header">
        <span>MODIS (Terra &amp; Aqua)</span>
        <span id="stat-0" style="color:#ff9f1c;">0 pts | 0 MW</span>
      </div>
      <div class="map-box"><div id="map-0" class="leaflet-map"></div></div>
    </div>

    <div class="map-card">
      <div class="card-header">
        <span>VIIRS NOAA-20</span>
        <span id="stat-1" style="color:#ff9f1c;">0 pts | 0 MW</span>
      </div>
      <div class="map-box"><div id="map-1" class="leaflet-map"></div></div>
    </div>

    <div class="map-card">
      <div class="card-header">
        <span>VIIRS NOAA-21</span>
        <span id="stat-2" style="color:#ff9f1c;">0 pts | 0 MW</span>
      </div>
      <div class="map-box"><div id="map-2" class="leaflet-map"></div></div>
    </div>
  </div>

<script>
  const SAT_KEYS = ["MODIS (Terra & Aqua)", "VIIRS NOAA-20", "VIIRS NOAA-21"];
  const SAT_DATA = {json.dumps(sat_payload)};
  const BBOX = [[{s}, {w}], [{n}, {e}]];

  const maps = [];
  const layerGroups = [];
  let isSyncing = false;

  SAT_KEYS.forEach((key, idx) => {{
    const m = L.map("map-" + idx, {{
      center: [{center_lat}, {center_lon}],
      zoom: 8,
      preferCanvas: true
    }});
    L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{{z}}/{{y}}/{{x}}", {{
      attribution: "Esri Dark Gray"
    }}).addTo(m);

    L.rectangle(BBOX, {{ color: "#888888", weight: 1.5, fill: false }}).addTo(m);
    m.fitBounds(BBOX);

    const lg = L.layerGroup().addTo(m);
    maps.push(m);
    layerGroups.push(lg);

    m.on("moveend", () => {{
      if (isSyncing) return;
      isSyncing = true;
      const c = m.getCenter();
      const z = m.getZoom();
      maps.forEach((other, j) => {{
        if (idx !== j) other.setView(c, z, {{ animate: false }});
      }});
      isSyncing = false;
    }});
  }});

  const monthSelect = document.getElementById("month-select");

  function renderCombined() {{
    const targetMonth = monthSelect.value;

    SAT_KEYS.forEach((key, idx) => {{
      const lg = layerGroups[idx];
      lg.clearLayers();

      const rows = SAT_DATA[key] || [];
      let count = 0;
      let totalFrp = 0;

      // Render up to 6,000 highest-FRP polygons per map cleanly via Canvas
      const filtered = rows.filter(r => targetMonth === "ALL" || r.m === targetMonth);
      const startIdx = Math.max(0, filtered.length - 6000);

      for (let i = 0; i < filtered.length; i++) {{
        const r = filtered[i];
        count++;
        totalFrp += r.frp;

        if (i >= startIdx) {{
          L.rectangle(r.b, {{
            color: r.c,
            weight: 0.4,
            opacity: r.o,
            fillColor: r.c,
            fillOpacity: r.o
          }}).bindPopup(
            `<b>${{key}}</b><br>Date: ${{r.d}}<br>FRP: <b>${{r.frp}} MW</b><br>Conf: <b>${{r.cl}}</b>`
          ).addTo(lg);
        }}
      }}

      document.getElementById("stat-" + idx).textContent =
        `${{count.toLocaleString()}} pts | ${{Math.round(totalFrp).toLocaleString()}} MW`;
    }});
  }}

  monthSelect.addEventListener("change", renderCombined);
  renderCombined();
</script>
</body>
</html>
"""

display(HTML(f'<iframe srcdoc="{html.escape(html_app)}" style="width:100%;height:540px;border:none;border-radius:8px;"></iframe>'))

/home/tinted/Projects/NASA-Space-Apps-2026/.venv/lib/python3.14/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [23]:
# @title Step 7A: Sensor-Agnostic Harmonization & 1.0 km Spatial Deduplication Engine
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from IPython.display import display

DEDUP_RADIUS_KM = 1.0  # @param {type:"number"}

def standardize_confidence(val) -> tuple[str, int]:
    """Maps MODIS (0-100) and VIIRS (l/n/h) confidence to ('Low'|'Nominal'|'High', rank 1..3)."""
    s = str(val).strip().lower()
    if s in ("l", "low"):
        return "Low", 1
    if s in ("n", "nominal"):
        return "Nominal", 2
    if s in ("h", "high"):
        return "High", 3
    try:
        num = float(s)
        if num < 30:
            return "Low", 1
        elif num < 80:
            return "Nominal", 2
        else:
            return "High", 3
    except ValueError:
        return "Nominal", 2

def harmonize_hotspots(sat_data_dict: dict[str, pd.DataFrame], radius_km: float = 1.0):
    """
    Combines MODIS, VIIRS NOAA-20, and VIIRS NOAA-21 detections into sensor-agnostic
    Harmonized 1 km Fire Events per Local Solar Fire Day.
    """
    frames = []
    for sat_label, df in sat_data_dict.items():
        if df is None or df.empty:
            continue
        sub = df.copy()
        sub["sensor_source"] = sat_label
        sub["short_sensor"] = (
            "MODIS" if "MODIS" in sat_label else ("NOAA-20" if "20" in sat_label else "NOAA-21")
        )
        sub["acq_time_utc"] = sub["acq_time"].astype(str).str.zfill(4)

        # Compute Local Solar Fire Day (UTC + lon/15 - 4h so afternoon D + overnight N group together)
        utc_dt = pd.to_datetime(
            sub["acq_date"].astype(str) + " " + sub["acq_time_utc"],
            format="%Y-%m-%d %H%M",
            utc=True,
        )
        solar_offset = pd.to_timedelta(sub["longitude"].astype(float) / 15.0 - 4.0, unit="h")
        sub["local_fire_date"] = (utc_dt + solar_offset).dt.strftime("%Y-%m-%d")

        conf_parsed = sub["confidence"].apply(standardize_confidence)
        sub["std_confidence"] = [c[0] for c in conf_parsed]
        sub["conf_rank"] = [c[1] for c in conf_parsed]
        sub["daynight_clean"] = sub["daynight"].astype(str).str.strip().str.upper()
        frames.append(sub)

    if not frames:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    raw_combined = pd.concat(frames, ignore_index=True)
    raw_combined["event_id"] = ""

    events = []
    next_cid = 0

    for fdate, group in raw_combined.groupby("local_fire_date", sort=True):
        # Anchor 1.0 km footprints on peak thermal cores first (FRP descending)
        g_sorted = group.sort_values("frp", ascending=False)
        idxs = g_sorted.index.to_numpy()
        lats = g_sorted["latitude"].to_numpy(dtype=float)
        lons = g_sorted["longitude"].to_numpy(dtype=float)

        # Equirectangular tangent-plane projection in kilometers for this fire day
        lat0_rad = np.radians(np.mean(lats))
        x_km = lons * 111.320 * np.cos(lat0_rad)
        y_km = lats * 111.320
        coords_km = np.column_stack([x_km, y_km])

        tree = cKDTree(coords_km)
        assigned = np.zeros(len(idxs), dtype=bool)

        for i in range(len(idxs)):
            if assigned[i]:
                continue
            nbrs = tree.query_ball_point(coords_km[i], r=radius_km)
            unassigned_nbrs = [j for j in nbrs if not assigned[j]]

            event_id = f"HFE-{fdate.replace('-', '')}-{next_cid:04d}"
            for j in unassigned_nbrs:
                assigned[j] = True
                raw_combined.loc[idxs[j], "event_id"] = event_id

            member_rows = g_sorted.iloc[unassigned_nbrs]
            weights = np.maximum(member_rows["frp"].to_numpy(dtype=float), 0.1)
            c_lat = float(np.average(member_rows["latitude"], weights=weights))
            c_lon = float(np.average(member_rows["longitude"], weights=weights))

            dn_set = set(member_rows["daynight_clean"])
            if "D" in dn_set and "N" in dn_set:
                persistence = "Persistent (Day + Night)"
            elif "N" in dn_set:
                persistence = "Night Only"
            else:
                persistence = "Day Only"

            # Sensor-calibrated FRP: average of per-sensor summed FRP inside the 1 km footprint
            per_sensor_frp = member_rows.groupby("short_sensor")["frp"].sum()
            calibrated_frp = float(per_sensor_frp.mean())
            raw_sum_frp = float(member_rows["frp"].sum())
            peak_frp = float(member_rows["frp"].max())

            max_rank = int(member_rows["conf_rank"].max())
            max_conf = {1: "Low", 2: "Nominal", 3: "High"}[max_rank]

            short_sensors = sorted(member_rows["short_sensor"].unique())

            events.append({
                "event_id": event_id,
                "local_fire_date": fdate,
                "latitude": round(c_lat, 5),
                "longitude": round(c_lon, 5),
                "persistence": persistence,
                "calibrated_frp_mw": round(calibrated_frp, 2),
                "peak_frp_mw": round(peak_frp, 2),
                "total_frp_mw": round(raw_sum_frp, 2),
                "max_confidence": max_conf,
                "raw_detections_merged": int(len(member_rows)),
                "sensor_count": int(len(short_sensors)),
                "corroborating_sensors": " + ".join(short_sensors),
                "has_modis": "MODIS" in short_sensors,
                "has_viirs": any("NOAA" in s for s in short_sensors),
                "first_utc": f"{member_rows['acq_date'].min()} {member_rows['acq_time_utc'].min()}",
                "last_utc": f"{member_rows['acq_date'].max()} {member_rows['acq_time_utc'].max()}",
            })
            next_cid += 1

    harmonized_df = pd.DataFrame(events)

    # Daily validation table comparing Raw Sensor Detections vs. Harmonized 1 km Events
    daily_summary = harmonized_df.groupby("local_fire_date").agg(
        harmonized_1km_events=("event_id", "count"),
        raw_detections_merged=("raw_detections_merged", "sum"),
        persistent_day_night=("persistence", lambda s: int((s == "Persistent (Day + Night)").sum())),
        multi_sensor_corroborated=("sensor_count", lambda s: int((s >= 2).sum())),
        calibrated_frp_mw=("calibrated_frp_mw", lambda s: round(s.sum(), 1)),
        uncalibrated_raw_frp_mw=("total_frp_mw", lambda s: round(s.sum(), 1)),
    )
    for short_s in ["MODIS", "NOAA-20", "NOAA-21"]:
        sub_s = raw_combined[raw_combined["short_sensor"] == short_s]
        daily_summary[f"raw_{short_s.lower().replace('-', '')}"] = sub_s.groupby("local_fire_date").size()

    daily_summary = daily_summary.fillna(0).reset_index()
    return harmonized_df, raw_combined, daily_summary

HARMONIZED_DF, RAW_COMBINED_DF, DAILY_VALIDATION_DF = harmonize_hotspots(SAT_DATA, radius_km=DEDUP_RADIUS_KM)

total_raw = len(RAW_COMBINED_DF)
total_harm = len(HARMONIZED_DF)
multi_sensor_pct = (HARMONIZED_DF["sensor_count"] >= 2).mean() * 100
modis_events = HARMONIZED_DF[HARMONIZED_DF["has_modis"]]
modis_viirs_agree_pct = (modis_events["has_viirs"].mean() * 100) if len(modis_events) else 0.0
persistent_pct = (HARMONIZED_DF["persistence"] == "Persistent (Day + Night)").mean() * 100

print(f"🔥 Raw Detections Input        : {total_raw:,} (MODIS + VIIRS NOAA-20 + VIIRS NOAA-21)")
print(f"🎯 Harmonized 1 km Fire Events : {total_harm:,} ({total_raw / total_harm:.2f}x spatial deduplication)")
print(f"🤝 Cross-Sensor Corroboration  : {multi_sensor_pct:.1f}% of events detected by ≥2 satellites ({modis_viirs_agree_pct:.1f}% of MODIS events corroborated by VIIRS)")
print(f"🌗 Day+Night Persistence Rate  : {persistent_pct:.1f}% of active 1 km fire cells burned across both Day & Night passes")
print(f"⚡ FRP Calibration Audit       : {HARMONIZED_DF['calibrated_frp_mw'].sum():,.1f} MW calibrated vs. {HARMONIZED_DF['total_frp_mw'].sum():,.1f} MW raw uncalibrated sum\n")

print("📊 Daily Raw vs. Harmonized Validation Table:")
display(DAILY_VALIDATION_DF)

print("\n🔍 Top 5 Harmonized Fire Events on Chosen Date:")
display(HARMONIZED_DF[HARMONIZED_DF["local_fire_date"] == CHOSEN_DATE].head(5))

🔥 Raw Detections Input        : 17,817 (MODIS + VIIRS NOAA-20 + VIIRS NOAA-21)
🎯 Harmonized 1 km Fire Events : 1,595 (11.17x spatial deduplication)
🤝 Cross-Sensor Corroboration  : 74.0% of events detected by ≥2 satellites (82.2% of MODIS events corroborated by VIIRS)
🌗 Day+Night Persistence Rate  : 24.6% of active 1 km fire cells burned across both Day & Night passes
⚡ FRP Calibration Audit       : 413,051.8 MW calibrated vs. 1,078,963.8 MW raw uncalibrated sum

📊 Daily Raw vs. Harmonized Validation Table:


,local_fire_date,harmonized_1km_events,raw_detections_merged,persistent_day_night,multi_sensor_corroborated,calibrated_frp_mw,uncalibrated_raw_frp_mw,raw_modis,raw_noaa20,raw_noaa21
0,2024-07-24,175,2582,1,136,60518.2,168918.4,270,895,1417
1,2024-07-25,409,5093,92,313,123834.0,341893.9,708,2092,2293
2,2024-07-26,513,4410,73,355,163189.8,388546.6,546,1708,2156
3,2024-07-27,126,1588,69,99,11539.3,31851.3,143,761,684
4,2024-07-28,146,1724,67,116,18599.9,53218.8,242,711,771
5,2024-07-29,107,1174,50,84,22346.5,59650.6,135,643,396
6,2024-07-30,83,1063,41,55,10634.4,29541.3,147,553,363
7,2024-07-31,36,183,0,22,2389.7,5342.7,16,104,63



🔍 Top 5 Harmonized Fire Events on Chosen Date:


,event_id,local_fire_date,latitude,longitude,persistence,calibrated_frp_mw,peak_frp_mw,total_frp_mw,max_confidence,raw_detections_merged,sensor_count,corroborating_sensors,has_modis,has_viirs,first_utc,last_utc
584,HFE-20240726-0584,2024-07-26,40.02684,-121.70238,Persistent (Day + Night),4620.13,6002.5,13860.40,High,49,3,MODIS + NOAA-20 + NOAA-21,True,True,2024-07-26 0852,2024-07-27 2216
585,HFE-20240726-0585,2024-07-26,40.34892,-121.93057,Night Only,1910.32,5414.9,5730.97,High,33,3,MODIS + NOAA-20 + NOAA-21,True,True,2024-07-27 0558,2024-07-27 1033
586,HFE-20240726-0586,2024-07-26,40.33432,-121.95667,Night Only,2562.86,5379.3,7688.58,High,27,3,MODIS + NOAA-20 + NOAA-21,True,True,2024-07-27 0558,2024-07-27 1033
587,HFE-20240726-0587,2024-07-26,40.01918,-121.67464,Persistent (Day + Night),5480.71,3824.1,10961.42,High,8,2,MODIS + NOAA-20,True,True,2024-07-26 0558,2024-07-27 2216
588,HFE-20240726-0588,2024-07-26,39.99738,-121.64790,Persistent (Day + Night),1385.82,3812.9,4157.45,High,24,3,MODIS + NOAA-20 + NOAA-21,True,True,2024-07-26 0558,2024-07-27 2216


In [24]:
# @title Step 7B: Side-by-Side Map — Raw Multi-Sensor Pixels vs. Harmonized 1 km Fire Events
import json
import html
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from IPython.display import display, HTML

w, s, e, n = map(float, CLEAN_BBOX.split(","))
center_lat, center_lon = (s + n) / 2.0, (w + e) / 2.0

avail_dates = sorted(HARMONIZED_DF["local_fire_date"].unique().tolist())
default_idx = avail_dates.index(CHOSEN_DATE) if CHOSEN_DATE in avail_dates else 0

frp_95 = float(np.percentile(HARMONIZED_DF["calibrated_frp_mw"], 95)) if not HARMONIZED_DF.empty else 500.0
cmap = plt.get_cmap("YlOrRd")

def frp_hex(val: float) -> str:
    norm = np.clip(val / max(frp_95, 1.0), 0.0, 1.0)
    return mcolors.to_hex(cmap(0.25 + 0.70 * norm))

# Pack Raw Detections (left map)
raw_records = []
for _, r in RAW_COMBINED_DF.sort_values("frp", ascending=True).iterrows():
    lat, lon = float(r["latitude"]), float(r["longitude"])
    res_m = 1000.0 if r["short_sensor"] == "MODIS" else 375.0
    h_dlat = (res_m / 111320.0) / 2.0
    h_dlon = (res_m / (111320.0 * max(np.cos(np.radians(lat)), 1e-6))) / 2.0
    raw_records.append({
        "d": r["local_fire_date"],
        "s": r["short_sensor"],
        "dn": r["daynight_clean"],
        "frp": round(float(r["frp"]), 1),
        "c": frp_hex(float(r["frp"])),
        "b": [[round(lat - h_dlat, 5), round(lon - h_dlon, 5)], [round(lat + h_dlat, 5), round(lon + h_dlon, 5)]],
    })

# Pack Harmonized 1 km Events (right map)
harm_records = []
for _, r in HARMONIZED_DF.sort_values("calibrated_frp_mw", ascending=True).iterrows():
    p_code = "P" if "Persistent" in r["persistence"] else ("D" if "Day" in r["persistence"] else "N")
    stroke = "#00f5d4" if p_code == "P" else ("#ff9f1c" if p_code == "D" else "#7aa2f7")
    harm_records.append({
        "id": r["event_id"],
        "d": r["local_fire_date"],
        "lat": float(r["latitude"]),
        "lon": float(r["longitude"]),
        "p": r["persistence"],
        "pc": p_code,
        "frp": float(r["calibrated_frp_mw"]),
        "raw_n": int(r["raw_detections_merged"]),
        "sens": r["corroborating_sensors"],
        "conf": r["max_confidence"],
        "c": frp_hex(float(r["calibrated_frp_mw"])),
        "st": stroke,
    })

html_val = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ background: #0e0e0e; color: #eee; font-family: -apple-system, BlinkMacSystemFont, monospace; padding: 10px; }}
    .toolbar {{
      background: #161616; border: 1px solid #333; border-radius: 6px; padding: 8px 14px;
      display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center; gap: 10px; margin-bottom: 10px; font-size: 12px;
    }}
    select {{
      background: #222; color: #00f5d4; border: 1px solid #00f5d4; padding: 4px 8px;
      border-radius: 4px; font-family: monospace; font-size: 12px; cursor: pointer;
    }}
    .grid-2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }}
    .card {{ background: #161616; border: 1px solid #333; border-radius: 6px; padding: 8px; }}
    .hdr {{ display: flex; justify-content: space-between; font-size: 12px; font-weight: bold; margin-bottom: 6px; }}
    .map-wrap {{ width: 100%; height: 420px; border-radius: 4px; overflow: hidden; }}
    .footer {{ margin-top: 10px; background: #161616; border: 1px solid #333; border-radius: 6px; padding: 10px 16px; }}
    input[type=range] {{ width: 100%; cursor: pointer; margin-top: 6px; }}
  </style>
</head>
<body>
  <div class="toolbar">
    <div>
      <b id="date-banner" style="color:#ff9f1c;">Local Fire Day: {avail_dates[default_idx]}</b>
      &nbsp;|&nbsp; Persistence Filter:
      <select id="p-filter">
        <option value="ALL">All Harmonized Events</option>
        <option value="P">Persistent (Day + Night) Only</option>
        <option value="D">Day Only</option>
        <option value="N">Night Only</option>
      </select>
    </div>
    <div>
      <span style="color:#00f5d4;">● Cyan Ring</span> = Persistent (Day+Night) &nbsp;
      <span style="color:#ff9f1c;">● Orange Ring</span> = Day Only &nbsp;
      <span style="color:#7aa2f7;">● Blue Ring</span> = Night Only
    </div>
  </div>

  <div class="grid-2">
    <div class="card">
      <div class="hdr">
        <span>LEFT: Raw Multi-Sensor Detections (MODIS + NOAA-20 + NOAA-21)</span>
        <span id="raw-stat" style="color:#ff9f1c;">0 raw pts</span>
      </div>
      <div id="map-raw" class="map-wrap"></div>
    </div>
    <div class="card">
      <div class="hdr">
        <span>RIGHT: Harmonized 1.0 km Fire Events (Sensor-Agnostic)</span>
        <span id="harm-stat" style="color:#00f5d4;">0 events</span>
      </div>
      <div id="map-harm" class="map-wrap"></div>
    </div>
  </div>

  <div class="footer">
    <div style="display:flex;justify-content:space-between;font-size:12px;">
      <span>📅 Scrub Local Solar Fire Day ({avail_dates[0]} → {avail_dates[-1]})</span>
      <span id=" compression-readout" style="color:#00f5d4;font-weight:bold;"></span>
    </div>
    <input id="d-slider" type="range" min="0" max="{len(avail_dates) - 1}" step="1" value="{default_idx}" />
  </div>

<script>
  const DATES = {json.dumps(avail_dates)};
  const RAW = {json.dumps(raw_records)};
  const HARM = {json.dumps(harm_records)};
  const BBOX = [[{s}, {w}], [{n}, {e}]];

  const mRaw = L.map("map-raw", {{ center: [{center_lat}, {center_lon}], zoom: 9, preferCanvas: true }});
  const mHarm = L.map("map-harm", {{ center: [{center_lat}, {center_lon}], zoom: 9, preferCanvas: true }});

  [mRaw, mHarm].forEach(m => {{
    L.tileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{{z}}/{{y}}/{{x}}", {{
      attribution: "Esri Dark Gray"
    }}).addTo(m);
    L.rectangle(BBOX, {{ color: "#666", weight: 1, fill: false }}).addTo(m);
    m.fitBounds(BBOX);
  }});

  const lgRaw = L.layerGroup().addTo(mRaw);
  const lgHarm = L.layerGroup().addTo(mHarm);
  let syncing = false;
  mRaw.on("moveend", () => {{ if (!syncing) {{ syncing = true; mHarm.setView(mRaw.getCenter(), mRaw.getZoom(), {{animate:false}}); syncing = false; }} }});
  mHarm.on("moveend", () => {{ if (!syncing) {{ syncing = true; mRaw.setView(mHarm.getCenter(), mHarm.getZoom(), {{animate:false}}); syncing = false; }} }});

  const dSlider = document.getElementById("d-slider");
  const pFilter = document.getElementById("p-filter");

  function updateMaps() {{
    const d = DATES[parseInt(dSlider.value, 10)];
    const pf = pFilter.value;
    document.getElementById("date-banner").textContent = "Local Fire Day: " + d;

    lgRaw.clearLayers();
    lgHarm.clearLayers();

    let rCount = 0, rFrp = 0;
    RAW.forEach(r => {{
      if (r.d !== d) return;
      rCount++;
      rFrp += r.frp;
      L.rectangle(r.b, {{ color: r.c, weight: 0.4, fillColor: r.c, fillOpacity: 0.55 }})
        .bindPopup(`<b>Raw ${{r.s}}</b><br>Date: ${{r.d}} (${{r.dn}})<br>FRP: ${{r.frp}} MW`)
        .addTo(lgRaw);
    }});

    let hCount = 0, hFrp = 0, mergedSum = 0;
    HARM.forEach(h => {{
      if (h.d !== d) return;
      if (pf !== "ALL" && h.pc !== pf) return;
      hCount++;
      hFrp += h.frp;
      mergedSum += h.raw_n;
      L.circle([h.lat, h.lon], {{
        radius: 1000,
        color: h.st,
        weight: 1.5,
        fillColor: h.c,
        fillOpacity: 0.72
      }}).bindPopup(
        `<b>${{h.id}}</b><br>Date: ${{h.d}}<br>Persistence: <b>${{h.p}}</b><br>` +
        `Calibrated FRP: <b>${{h.frp}} MW</b><br>Raw Merged: <b>${{h.raw_n}} pts</b><br>` +
        `Sensors: <b>${{h.sens}}</b><br>Max Conf: ${{h.conf}}`
      ).addTo(lgHarm);
    }});

    document.getElementById("raw-stat").textContent = `${{rCount.toLocaleString()}} pts | ${{Math.round(rFrp).toLocaleString()}} MW raw`;
    document.getElementById("harm-stat").textContent = `${{hCount.toLocaleString()}} events (1 km) | ${{Math.round(hFrp).toLocaleString()}} MW cal`;
  }}

  dSlider.addEventListener("input", updateMaps);
  pFilter.addEventListener("change", updateMaps);
  updateMaps();
</script>
</body>
</html>
"""

display(HTML(f'<iframe srcdoc="{html.escape(html_val)}" style="width:100%;height:540px;border:none;border-radius:8px;"></iframe>'))

/home/tinted/Projects/NASA-Space-Apps-2026/.venv/lib/python3.14/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [25]:
# @title Step 7C: Export & Download Standardized Harmonized Hotspots CSV
import pandas as pd

def classify_intensity(frp_mw: float) -> str:
    if frp_mw < 10.0:
        return "Low (<10 MW)"
    elif frp_mw < 50.0:
        return "Moderate (10-50 MW)"
    elif frp_mw < 250.0:
        return "High (50-250 MW)"
    return "Extreme (250+ MW)"

def export_harmonized_csv(raw_combined_df: pd.DataFrame, harmonized_df: pd.DataFrame, out_filename: str = "harmonized_fire_hotspots.csv") -> pd.DataFrame:
    # Compute true chronological UTC start/end per event_id
    utc_series = pd.to_datetime(
        raw_combined_df["acq_date"].astype(str) + " " + raw_combined_df["acq_time_utc"].astype(str),
        format="%Y-%m-%d %H%M",
        utc=True,
    )
    time_bounds = raw_combined_df.assign(utc_dt=utc_series).groupby("event_id")["utc_dt"].agg(["min", "max"])

    export_df = pd.DataFrame({
        "fire_event_id": harmonized_df["event_id"],
        "fire_date_local": harmonized_df["local_fire_date"],
        "first_detected_utc": harmonized_df["event_id"].map(time_bounds["min"]).dt.strftime("%Y-%m-%d %H:%M"),
        "last_detected_utc": harmonized_df["event_id"].map(time_bounds["max"]).dt.strftime("%Y-%m-%d %H:%M"),
        "latitude": harmonized_df["latitude"],
        "longitude": harmonized_df["longitude"],
        "fire_persistence": harmonized_df["persistence"],
        "fire_power_mw": harmonized_df["calibrated_frp_mw"],
        "peak_pixel_power_mw": harmonized_df["peak_frp_mw"],
        "intensity_category": harmonized_df["calibrated_frp_mw"].apply(classify_intensity),
        "confidence_level": harmonized_df["max_confidence"],
        "detections_merged": harmonized_df["raw_detections_merged"],
        "sensor_count": harmonized_df["sensor_count"],
        "detecting_sensors": harmonized_df["corroborating_sensors"],
    })

    export_df.to_csv(out_filename, index=False)
    print(f"✅ Exported {len(export_df):,} harmonized fire events to '{out_filename}'")
    try:
        from google.colab import files
        files.download(out_filename)
    except Exception:
        print("ℹ️ Saved locally (auto-download triggers when running inside Google Colab).")
    return export_df

EXPORT_CSV_DF = export_harmonized_csv(RAW_COMBINED_DF, HARMONIZED_DF, f"harmonized_fire_hotspots_{CHOSEN_DATE}.csv")
display(EXPORT_CSV_DF.head(5))

✅ Exported 1,595 harmonized fire events to 'harmonized_fire_hotspots_2024-07-26.csv'
ℹ️ Saved locally (auto-download triggers when running inside Google Colab).


,fire_event_id,fire_date_local,first_detected_utc,last_detected_utc,latitude,longitude,fire_persistence,fire_power_mw,peak_pixel_power_mw,intensity_category,confidence_level,detections_merged,sensor_count,detecting_sensors
0,HFE-20240724-0000,2024-07-24,2024-07-25 04:38,2024-07-25 10:20,39.90980,-121.82287,Night Only,3312.87,5544.5,Extreme (250+ MW),High,15,3,MODIS + NOAA-20 + NOAA-21
1,HFE-20240724-0001,2024-07-24,2024-07-25 06:16,2024-07-25 11:10,39.90197,-121.83901,Night Only,2237.53,3709.7,Extreme (250+ MW),High,41,3,MODIS + NOAA-20 + NOAA-21
2,HFE-20240724-0002,2024-07-24,2024-07-25 04:38,2024-07-25 11:10,39.89520,-121.81593,Night Only,1916.95,3353.2,Extreme (250+ MW),High,31,3,MODIS + NOAA-20 + NOAA-21
3,HFE-20240724-0003,2024-07-24,2024-07-25 06:16,2024-07-25 11:10,39.95346,-121.84439,Night Only,2060.09,3308.7,Extreme (250+ MW),High,35,3,MODIS + NOAA-20 + NOAA-21
4,HFE-20240724-0004,2024-07-24,2024-07-25 04:38,2024-07-25 11:10,39.91921,-121.78296,Night Only,2473.41,3063.6,Extreme (250+ MW),High,36,3,MODIS + NOAA-20 + NOAA-21
